<a href="https://colab.research.google.com/github/ahmadhalawanii/flyrank-ml-capstone/blob/main/w01_research_question_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmadhalawanii/flyrank-ml-capstone/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (Colab or local)
On Colab this clones the repo, installs requirements, and runs the pipeline once so
`data/processed/model_predictions.csv` exists (it's gitignored -- generated, not shipped).

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ahmadhalawanii/flyrank-ml-capstone"  # your fork
REPO_DIR = "flyrank-ml-capstone"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # local run: walk up from wherever this notebook sits (e.g. work/notebooks/) to the repo root
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found -- are you at the repo root?"

# data/processed/ is gitignored (it's regenerated), so run the pipeline once this session
if not os.path.exists("data/processed/model_predictions.csv"):
    print("Running the pipeline once to generate model_predictions.csv (~1 minute)...")
    subprocess.run([sys.executable, "scripts/run_all.py"], check=True)

assert os.path.exists("data/processed/model_predictions.csv"), "pipeline did not produce predictions -- check the run_all.py output above"
print("Setup complete -- data and predictions are ready.")

Working dir: /content/flyrank-ml-capstone
Running the pipeline once to generate model_predictions.csv (~1 minute)...
Setup complete -- data and predictions are ready.


## 1. My lane (or freestyle) and why

**Freestyle.**

Question: does the pipeline's Precision@50 score actually cover every content type it ranks, or
does it hide some of them?

I checked the real test-set predictions instead of trusting the one summary number. Two problems
showed up:

- `comparison article` pages (697 of them) all belong to one client. That client landed in
  training, not testing. So the reported score has zero evidence for this content type.
- `feedly article` pages score 0.68 on their own. The reported 0.84 never shows this, because
  `keyword article` pages dominate the ranking and hide the gap.

None of the four lane guide lanes ask this question directly -- they all ask "build a ranking."
This asks: is the ranking's own reported score actually true for everything it ranks?

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

freestyle_question = (
    "Does the pipeline's reported Precision@50 actually cover every content type "
    "it's used to rank -- or are some segments unmeasured or hidden inside the average?"
)
print(freestyle_question)

Does the pipeline's reported Precision@50 actually cover every content type it's used to rank -- or are some segments unmeasured or hidden inside the average?


## 2. The question: decision, action, cost of a wrong call

**Decision:** can a content strategist trust this cycle's review queue for every content type, or
does one type need a manual check first?

**Unit of analysis:** a page, grouped by `content_type` -- that grouping is what exposed the
problem.

**Who acts:** the strategist who pulls the review queue each sprint.

**Action:** before trusting the queue, check if every content type was actually covered by the
test set. If a type has zero coverage (comparison articles) or a real score below the reported
number (feedly articles), review it by hand this cycle instead of trusting the ranking.

**Cost of a wrong call:** a whole content type could be ranked badly and nothing would show it,
because the dashboard's one number cannot see that type. A genuinely declining page slips through
the queue while the reported score still looks fine.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

# The mechanism behind the comparison-article blind spot: how many clients
# even publish that type, and did any of them land in the test split?
import pandas as pd

raw = pd.read_csv("data/raw/content_refresh_anonymized.csv")
preds = pd.read_csv("data/processed/model_predictions.csv")

comparison_clients = set(raw.loc[raw["content_type"] == "comparison article", "client_id"].unique())
test_clients = set(preds.loc[preds["split"] == "test", "client_id"].unique())

print(f"Clients publishing comparison articles: {len(comparison_clients)} of {raw['client_id'].nunique()}")
print(f"Of those, landed in the test holdout: {len(comparison_clients & test_clients)}")
print("-> one client owns the whole content type, and the random client-level holdout")
print("   happened to keep that client in training this run.")

Clients publishing comparison articles: 1 of 32
Of those, landed in the test holdout: 0
-> one client owns the whole content type, and the random client-level holdout
   happened to keep that client in training this run.


## 3. Quick look at the data (2-3 real numbers)

1. **Comparison articles: 697 pages, 1 client, 0 test rows.** The model's reported score has zero
   evidence for this content type.
2. **Feedly articles: real Precision@50 = 0.68**, vs. the reported overall 0.84.
3. **Overall Precision@50 (0.84) = keyword-article-only Precision@50 (0.84).** The "overall"
   number is really just the biggest group's number.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

import pandas as pd, numpy as np

raw = pd.read_csv("data/raw/content_refresh_anonymized.csv")
preds = pd.read_csv("data/processed/model_predictions.csv")
df = preds.merge(raw, on=["content_id", "client_id"], how="left")
test = df[df["split"] == "test"].copy()

def precision_at_k(scores, labels, k):
    k = min(k, len(scores))
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean(), k

overall_p, k = precision_at_k(test["prob_random_forest"], test["is_declining_label"], 50)
print(f"Overall test Precision@{k}: {overall_p:.3f}\n")

print("By content_type (test split only):")
for ct, g in test.groupby("content_type"):
    p, kk = precision_at_k(g["prob_random_forest"], g["is_declining_label"], 50)
    print(f"  {ct:20s} n={len(g):4d}  base_rate={g['is_declining_label'].mean():.3f}  "
          f"Precision@{kk}={p:.3f}")

missing = set(raw["content_type"].unique()) - set(test["content_type"].unique())
print(f"\nContent types with ZERO test-set rows: {missing if missing else 'none'}")

Overall test Precision@50: 0.840

By content_type (test split only):
  feedly article       n= 958  base_rate=0.201  Precision@50=0.680
  keyword article      n=1367  base_rate=0.524  Precision@50=0.840

Content types with ZERO test-set rows: {'comparison article'}


## 4. Careful words: what I can and can't claim

**Can say:**

- On this run, one content type had zero test coverage, and another scored meaningfully worse
  than the reported number.
- This is a weakness of the evaluation split, not of the random forest itself -- any model
  trained on this same split would have the same blind spot.
- A coverage check like this should run before anyone trusts a model's headline score.

**Can't say:**

- That the model is "bad" at comparison articles -- there is no evidence either way, that's the
  point.
- That this exact blind spot is permanent -- a different random split could hide a different
  content type instead.
- Anything about a real client, URL, or query. The data stays anonymized.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

# Public-safety check: confirm no identifying columns snuck into the working frame.
unsafe_field_names = ["client_name", "domain", "url", "query", "keyword", "title"]
present = [c for c in df.columns if any(term in c.lower() for term in unsafe_field_names)]
print("Potentially unsafe columns present:", present if present else "none")
assert not present, "Found a column that should not be here -- stop and check before publishing."


Potentially unsafe columns present: none


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.